# Generating Table 1 using connectivity and birth time data

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import networkx.algorithms.community as nx_comm
import pickle
import time
import random
import math
import pandas as pd

In [3]:
stage_labels = ['L1-0', 'L1-5', 'L1-8', 'L1-16', 'L2', 'L3', '$Ad_{1}$', '$Ad_{2}$']

In [4]:
stage_labels_plot = ['L1$_{0h}$', 'L1$_{5h}$', 'L1$_{8h}$', 'L1$_{16h}$', 'L2', 'L3', 'Ad$^{1}$', 'Ad$^{2}$']

**Generating graphs per stage by incorporating isolates based on the birth time data**

In [5]:
with open('data/processed/all_c_elegans_weighted_graphs.pkl', 'rb') as file:
    all_c_elegans_weighted_graphs = pickle.load(file)

In [6]:
stage_times = [800, 1100, 1280, 1760, 2180, 2420, 3500, 3500] # lower estimates from Witvliet Paper

In [10]:
df = pd.read_csv("data/raw/birth_times_for_worm_brain.csv")

birth_times_dict_witvliet_allcells = dict(zip(df["cell"], df["birth time"]))

In [13]:
union_graph = nx.compose_all(all_c_elegans_weighted_graphs) 
print(union_graph)

DiGraph with 223 nodes and 3665 edges


In [14]:
d = dict([(tup[0], tup[1]['specific_class']) for tup in list(union_graph.nodes(data=True))])
spclass_to_idx = {v:k for k,v in d.items()}

In [15]:
all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included = []
for idx, graph in enumerate(all_c_elegans_weighted_graphs):
    stg_time = stage_times[idx]
    alreadyborn = {neu for neu, time in birth_times_dict_witvliet_allcells.items() if time <= stg_time}
    neurons_in_graph = {graph.nodes[node]['specific_class'] for node in graph.nodes()}
    for neuron in alreadyborn:
        if neuron not in neurons_in_graph:
            index = spclass_to_idx[neuron]
            attr = union_graph.nodes[index]
            graph.add_node(index, **attr)
            #print(idx, neuron, index)
    print(stg_time)
    all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included.append(graph)
            

800
1100
1280
1760
2180
2420
3500
3500


In [16]:
list(nx.selfloop_edges(all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included[6], data=True)) # all have low weights

[]

In [83]:
d = all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included[0].nodes[0]['parent_class_info']

In [17]:
for graph in all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included:
    celltypes = set()
    for cell in graph.nodes():
        celltypes.add(graph.nodes[cell]['parent_class_info'].get('celltype', 'NA'))
    print(celltypes)

{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}
{'Sensory neuron', 'Muscle', 'Motor neuron', 'Interneuron', 'NA', 'Glia', 'Modulatory neuron'}


In [18]:
def mean_weight(graph):
    return np.mean([d['weight'] for _, _, d in graph.edges(data=True)])

In [19]:
for graph in all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included:
    print(mean_weight(graph))

1.6722580645161291
1.9219066937119675
2.102766798418972
2.444542253521127
2.716831683168317
2.9219672131147543
3.402555910543131
3.645928636779506


**Most neurons of the head/brain/nerve ring are present at birth**

In [20]:
for graph in all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included:
    neurons = 0
    for cell in graph.nodes():
        if graph.nodes[cell]['parent_class_info'].get('celltype', 'NA') not in {'Glia', 'Muscle'}:
            neurons += 1 
    print(neurons)

167
167
168
177
180
181
181
181


In [22]:
# for paper table
for graph in all_c_elegans_weighted_graphs_isolated_neurons_per_birthtime_included:
    neurons = []
    for cell in graph.nodes():
        if graph.nodes[cell]['parent_class_info'].get('celltype', 'NA') not in {'Glia', 'Muscle'}: # CANR/CANL will be correctly taken as neurons since their celltype is NA
            neurons.append(cell)
            
    all_cells = graph.number_of_nodes()
    edges = graph.number_of_edges()
    density = round(nx.density(graph), 4)
    avg_edge_weight = round(mean_weight(graph), 3)
    neuronal_subgraph = nx.subgraph(graph, neurons)
    neuronal_density = round(nx.density(neuronal_subgraph), 5)
    
    # weak/strong connectivity analyses done on the graph EXCLUDING isolated nodes else even weak connectivity would be false
    graph_no_isolates = graph.subgraph(n for n, d in graph.degree() if d > 0).copy()
    wcc = nx.is_weakly_connected(graph_no_isolates)
    scc = nx.is_strongly_connected(graph_no_isolates)
    SCC_fraction = max(map(len, nx.strongly_connected_components(graph_no_isolates))) / graph_no_isolates.number_of_nodes()

    print(
    "Nc =", all_cells,
    "| Nn =", len(neurons),
    "| E =", edges,
    "| S =", sum([d['weight'] for _, _, d in graph.edges(data=True)]),
    "| ρ =", density,
    "| ρn =", neuronal_density,
    "| ⟨k⟩ =", round(np.mean(list(dict(graph.degree()).values())), 2),
    "| |WC| =", wcc,
    "| |SC| =", scc,
    "| ⟨w⟩ =", avg_edge_weight
    )

Nc = 209 | Nn = 167 | E = 775 | S = 1296 | ρ = 0.0178 | ρn = 0.02435 | ⟨k⟩ = 7.42 | |WC| = True | |SC| = False | ⟨w⟩ = 1.672
Nc = 209 | Nn = 167 | E = 986 | S = 1895 | ρ = 0.0227 | ρn = 0.0312 | ⟨k⟩ = 9.44 | |WC| = True | |SC| = False | ⟨w⟩ = 1.922
Nc = 210 | Nn = 168 | E = 1012 | S = 2128 | ρ = 0.0231 | ρn = 0.03162 | ⟨k⟩ = 9.64 | |WC| = True | |SC| = False | ⟨w⟩ = 2.103
Nc = 219 | Nn = 177 | E = 1136 | S = 2777 | ρ = 0.0238 | ρn = 0.03245 | ⟨k⟩ = 10.37 | |WC| = True | |SC| = False | ⟨w⟩ = 2.445
Nc = 222 | Nn = 180 | E = 1515 | S = 4116 | ρ = 0.0309 | ρn = 0.04109 | ⟨k⟩ = 13.65 | |WC| = True | |SC| = False | ⟨w⟩ = 2.717
Nc = 223 | Nn = 181 | E = 1525 | S = 4456 | ρ = 0.0308 | ρn = 0.04036 | ⟨k⟩ = 13.68 | |WC| = True | |SC| = False | ⟨w⟩ = 2.922
Nc = 223 | Nn = 181 | E = 2191 | S = 7455 | ρ = 0.0443 | ρn = 0.05933 | ⟨k⟩ = 19.65 | |WC| = True | |SC| = False | ⟨w⟩ = 3.403
Nc = 223 | Nn = 181 | E = 2186 | S = 7970 | ρ = 0.0442 | ρn = 0.05933 | ⟨k⟩ = 19.61 | |WC| = True | |SC| = False | ⟨w